# 1 - Import

In [ ]:
!pip install split-folders

In [ ]:
import kagglehub
import pathlib
import tensorflow as tf
import matplotlib.pyplot as plt
import splitfolders
import shutil
import os

# 2 - Download Dati

## 2.1 - Download da kagglehub

In [ ]:
path = pathlib.Path(kagglehub.dataset_download("jehanbhathena/weather-dataset"))

print("Path to dataset files:", path)

## 2.2 - Spostamento e Rimozione file superlui

In [ ]:
DATA_PATH = pathlib.Path("./data")

if DATA_PATH.exists():
    shutil.rmtree(DATA_PATH)

shutil.copytree(path, DATA_PATH)

In [ ]:
def find_invalid_images(directory):
    invalid_files = []
    for root, _, files in os.walk(directory):
        for f in files:
            file_path = os.path.join(root, f)
            try:
                img_bytes = tf.io.read_file(file_path)
                _ = tf.image.decode_image(img_bytes)
            except tf.errors.InvalidArgumentError:
                invalid_files.append(file_path)
            except Exception as e:
                print(f"An unexpected error occurred with file {file_path}: {e}")
                invalid_files.append(file_path)
    return invalid_files

def remove_files(file_list):
    for file_path in file_list:
        try:
            os.remove(file_path)
            print(f"Removed: {file_path}")
        except OSError as e:
            print(f"Error removing file {file_path}: {e}")

In [ ]:
invalid_files = find_invalid_images(DATA_PATH)
remove_files(invalid_files)

# 3 - Split in Train Test e Validation

## 3.1 - Caricamento dati

## 3.2 - Suddivisione

In [ ]:
DATA_FILES = pathlib.Path("./data/dataset")

splitfolders.ratio(DATA_FILES, output="diviso", seed=42, ratio=(.8, .1, .1))

In [ ]:
diviso = pathlib.Path("./diviso")

train_dir = diviso / "train"
val_dir = diviso / "val"
test_dir = diviso / "test"

SEED = 42
BATCH_SIZE = 16
IMG_SIZE = (242, 242)

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    shuffle = True,
    seed = SEED
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    seed = SEED
)


test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    seed = SEED
)


# 4 - Esplorazione Dataset

## 4.1 Visualizzazione immagini

In [ ]:
class_names = train_dataset.class_names

plt.figure(figsize=(10,10))

for images, labels in train_dataset.take(1):
  for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")
plt.show()

## 4.2 Numeriche Del dataset (n. classi, nomi classi ecc)



In [ ]:
print("Numero di classi: ", len(class_names))
print("Nomi classi: ", class_names)

# 5 - Preprocessing (se TF, normalizzazione e cache + prefetch)

In [ ]:
preprocess_input = tf.keras.applications.efficientnet.preprocess_input

train_dataset  = train_dataset.cache().prefetch(2)
validation_dataset = validation_dataset.cache().prefetch(2)
test_dataset = test_dataset.cache().prefetch(2)

# 6 - Definizione del Modello

In [ ]:
backbone = tf.keras.applications.EfficientNetB4(
    include_top = False,
    weights = "imagenet",
    input_shape = IMG_SIZE + (3,)
)

backbone.trainable = False

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
])

In [ ]:
classification_heads = tf.keras.models.Sequential(
    [
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128, activation = "relu"),
        tf.keras.layers.Dense(11, activation = "softmax")
    ]
)

# 7 - Valutazione performance

In [ ]:
inputs = tf.keras.layers.Input(shape = IMG_SIZE + (3,))
x = preprocess_input(inputs)
x = data_augmentation(x)
x = backbone(x)
outputs = classification_heads(x)
model = tf.keras.Model(inputs, outputs)

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

learning_rate_adapter = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3
)

checkpoint_model = tf.keras.callbacks.ModelCheckpoint(
    filepath="./best.keras",
    monitor="val_loss",
    save_best_only=True
)

checkpoint_weights = tf.keras.callbacks.ModelCheckpoint(
    filepath="./best.weights.h5",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True
)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate = 0.0001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=100,
    callbacks=[
        early_stopping,
        learning_rate_adapter,
        checkpoint_model,
        checkpoint_weights
    ]
)

## 7.1 Curve di apprendimento

In [ ]:
plt.plot(history.history['loss'], label="Training Loss")
plt.plot(history.history['val_loss'], label="Validation Loss")
plt.legend()
plt.xlabel("Numero di Epoche")
plt.ylabel("Valore Loss")
plt.title("Loss durante l'addestramento")
plt.show()

In [ ]:
plt.plot(history.history['accuracy'], label="Training Accuracy")
plt.plot(history.history['val_accuracy'], label="Validation Accuracy")
plt.legend()
plt.xlabel("Numero di Epoche")
plt.ylabel("Valore Accuracy")
plt.title("Accuracy durante l'addestramento")
plt.show()

# 7.2 Metriche su Test Set

In [ ]:
best_model = tf.keras.models.load_model("./best.keras")

model.load_weights("./best.weights.h5")

results = model.evaluate(test_dataset)
print(f'Accuracy : {results[1]}')